# Day 2, Segment 1: Building a Complete RAG Pipeline

## Overview
In this segment, we'll build a **Retrieval-Augmented Generation (RAG)** pipeline that combines document ingestion, semantic search, and LLM-powered answer generation.

### What is RAG?
RAG is a pattern that:
1. **Retrieves** relevant documents from a knowledge base
2. **Augments** the LLM prompt with those documents
3. **Generates** contextually accurate answers

### Pipeline Architecture
```
📄 Documents → 🔄 Chunking → 🧮 Embeddings → 📊 Vector Store
                                                      ↓
🎯 Query → 🔍 Retrieval → 📝 Context Injection → 🤖 LLM Answer
```

### Key Components
- **Document Loader**: Extract text from files (PDF, TXT, Markdown, etc.)
- **Text Splitter**: Break documents into manageable chunks
- **Embeddings**: Convert text to numerical vectors for semantic search
- **Vector Store**: Store embeddings for fast similarity search
- **Retriever**: Find relevant documents based on query similarity
- **LLM**: Generate answers using retrieved context

Let's build this step-by-step!

In [69]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ Environment loaded")

✓ Environment loaded


## Step 1: Document Ingestion

**Goal**: Load raw documents from files into LangChain Document objects.

**Key Concepts**:
- LangChain provides loaders for various formats (TextLoader, PyPDFLoader, MarkdownLoader, etc.)
- Each document has `page_content` (text) and `metadata` (source, page number, etc.)

**Production Pattern**: Extract-Transform-Load (ETL)
- Extract: Load raw documents
- Transform: Add metadata, validate content
- Load: Prepare for next stage

In [70]:
from langchain_community.document_loaders import TextLoader
import os

def load_documents():
    """
    Load documents from text files.
    
    Returns:
        List of LangChain Document objects with page_content and metadata
    """
    loader = TextLoader("data/notes.txt")   # replace with your files
    return loader.load()

## Step 2: Text Chunking

**Goal**: Split large documents into smaller, semantically meaningful chunks.

**Why Chunking?**
- LLMs have context window limits (e.g., 128k tokens for GPT-4o)
- Smaller chunks improve retrieval relevance
- Optimal chunk size depends on domain (500-1500 chars typical)

**Chunking Strategy**:
- `chunk_size`: Number of characters per chunk
- `chunk_overlap`: Overlap to preserve context across boundaries
- Example: 500 chars with 100 overlap = 400 new chars per chunk

In [71]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents(docs):
    """
    Split documents into chunks using recursive character splitting.
    
    Args:
        docs: List of Document objects
        
    Returns:
        List of smaller Document chunks
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,      # Characters per chunk
        chunk_overlap=100    # Overlap for context preservation
    )

    return splitter.split_documents(docs)

## Step 3: Generate Embeddings

**Goal**: Convert text chunks into numerical vectors for semantic search.

**Key Concepts**:
- **Embeddings**: Dense numerical representations of text
- **Semantic Search**: Find similar documents based on meaning, not keywords
- **Model Choice**: text-embedding-3-small (lightweight, recommended by OpenAI)

**How It Works**:
- Input: Text chunk
- Process: Neural network converts to 1536-dimensional vector
- Output: Vector captures semantic meaning of the text

**Production Pattern**: 
- Use consistent embedding model for all documents
- Store embeddings in vector database with document references

In [72]:
from langchain_openai import OpenAIEmbeddings

def create_embeddings():
    """
    Create an embeddings model using OpenAI's text-embedding-3-small.
    
    Returns:
        OpenAIEmbeddings instance configured with API key
    """
    return OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

## Step 4: Create Vector Store

**Goal**: Store embeddings in a database for fast semantic search.

**Why Vector Stores?**
- Traditional keyword search: "enterprise" doesn't match "large organization"
- Vector search: Semantically similar concepts have nearby vectors
- Fast similarity queries: Millions of documents in milliseconds

**Chroma Vector Store**:
- Lightweight, in-memory by default
- Supports persistent storage
- Handles document metadata alongside embeddings
- Built-in similarity search methods

**Key Operations**:
- `from_documents()`: Create store from document chunks + embeddings
- `similarity_search()`: Find k most similar documents
- `collection_name`: Organize documents into named collections

In [73]:
from langchain_community.vectorstores import Chroma

def create_vector_store(chunks, embeddings):
    """
    Create a Chroma vector store from document chunks and embeddings.
    
    Args:
        chunks: List of Document chunks
        embeddings: OpenAIEmbeddings instance
        
    Returns:
        Chroma VectorStore instance
    """
    return Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="rag_demo"
    )

## Step 5: Retrieve Relevant Context

**Goal**: Find the most relevant documents for a given query.

**Retrieval Process**:
1. Convert query to embedding using same model
2. Compare with stored document embeddings
3. Return k most similar documents (by cosine distance)

**Parameters**:
- `query`: User question or information need
- `k`: Number of documents to return (typically 3-5)

**Quality Metrics**:
- **Precision**: Are returned documents relevant?
- **Recall**: Are all relevant documents included?
- Trade-off: Higher k = better recall but more noise

In [74]:
def build_context(docs):
    """
    Combine retrieved documents into a single context string.
    
    Args:
        docs: List of Document objects
        
    Returns:
        String with document contents separated by double newlines
    """
    return "\n\n".join([doc.page_content for doc in docs])

In [75]:
def retrieve_context(vector_store, query, k=3):
    """
    Retrieve the k most relevant documents for a query.
    
    Args:
        vector_store: Chroma VectorStore instance
        query: User query string
        k: Number of documents to retrieve (default: 3)
        
    Returns:
        List of relevant Document objects
    """
    return vector_store.similarity_search(query, k=k)

## Step 6: Context Injection & LLM Generation

**Goal**: Use retrieved context to generate accurate, grounded answers.

**Why Context Injection?**
- Prevents hallucinations: LLM references actual documents
- Reduces outdated information: Uses latest knowledge base
- Enables domain expertise: Documents contain specialized information

**Prompt Structure**:
```
System: "You are a helpful assistant."
User: "Use the following context to answer the question.
      Context: [RETRIEVED_DOCUMENTS]
      Question: [USER_QUERY]"
```

**LLM Configuration**:
- `temperature=0`: Deterministic responses (no randomness)
- `model="gpt-4o"`: Latest OpenAI model with strong reasoning

**Key Pattern**: Separate concerns
- Context retrieval: Vector store logic
- Answer generation: LLM + prompt engineering

In [76]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

def generate_answer(context, query):
    """
    Generate an answer using retrieved context and the LLM.
    
    Args:
        context: String containing retrieved documents
        query: User query
        
    Returns:
        Generated answer string
    """
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0,  # Deterministic responses
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

    messages = [
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}
""")
    ]

    return llm.invoke(messages).content

## Step 7: Complete RAG Pipeline

**Goal**: Orchestrate all components into a single reusable function.

**Pipeline Flow**:
1. **Ingestion**: Load documents from files
2. **Chunking**: Split into manageable pieces
3. **Embeddings**: Convert to vectors
4. **Storage**: Create vector store
5. **Retrieval**: Find relevant documents
6. **Context**: Build context string
7. **Generation**: LLM produces answer

**Orchestration Pattern**:
- Single entry point (`rag_pipeline`)
- Clear function composition
- Each step can be tested independently
- Easy to swap components

**Error Handling Considerations**:
- Document loading: File not found?
- API calls: Rate limits, timeouts?
- Vector operations: Dimension mismatches?

This modular approach enables robust, maintainable production systems!

In [77]:
def rag_pipeline(query):
    """
    Complete RAG pipeline from document to answer.
    
    Args:
        query: User question
        
    Returns:
        Generated answer based on retrieved context
    """
    # Step 1: Ingestion
    docs = load_documents()
    print(f"✓ Loaded {len(docs)} document(s)")

    # Step 2: Chunking
    chunks = chunk_documents(docs)
    print(f"✓ Created {len(chunks)} chunk(s)")

    # Step 3: Embeddings
    embeddings = create_embeddings()
    print("✓ Embeddings model ready")

    # Step 4: Storage
    vector_store = create_vector_store(chunks, embeddings)
    print("✓ Vector store created")

    # Step 5: Retrieval
    retrieved_docs = retrieve_context(vector_store, query, k=3)
    print(f"✓ Retrieved {len(retrieved_docs)} document(s)")

    # Step 6: Context Injection
    context = build_context(retrieved_docs)

    # Step 7: LLM
    answer = generate_answer(context, query)
    print("✓ Answer generated")

    return answer

## Step 8: Test the RAG Pipeline

**Scenario**: Question answering about pricing strategies

This test demonstrates the complete flow:
- Query: Natural language question
- Retrieval: Find relevant documents about pricing
- Generation: Answer based on retrieved context

**Expected Behavior**:
- Each step prints progress markers
- Final output: AI-generated answer grounded in documents

In [78]:
query = "What pricing strategy works best for enterprise customers?"

print(f"📝 Query: {query}\n")

response = rag_pipeline(query)

print(f"\n💡 Answer:\n{response}")

📝 Query: What pricing strategy works best for enterprise customers?

✓ Loaded 1 document(s)
✓ Created 6 chunk(s)
✓ Embeddings model ready
✓ Vector store created
✓ Retrieved 3 document(s)
✓ Answer generated

💡 Answer:
The best pricing strategy for enterprise customers is to offer higher-priced tiers that prioritize reliability, security, and support. Enterprise customers are willing to pay higher prices for guaranteed uptime and dedicated account management, so a tiered pricing strategy that includes these features would be effective.


## Key Takeaways

### 1. **RAG Pattern Overview**
- Retrieves relevant documents before generation
- Grounds LLM responses in actual data
- Reduces hallucinations and improves accuracy

### 2. **Component Specialization**
- **Loaders**: Extract from various sources (PDF, TXT, web, databases)
- **Splitters**: Chunk strategically based on domain knowledge
- **Embeddings**: Semantic understanding through vectors
- **Vector Stores**: Fast similarity search at scale
- **LLMs**: Answer generation with context injection

### 3. **Production Patterns**
- **Modularity**: Each component is testable and replaceable
- **Error Handling**: Graceful degradation when retrieval fails
- **Performance**: Vector search scales to millions of documents
- **Monitoring**: Track retrieval quality and generation accuracy

### 4. **Common Challenges & Solutions**
| Challenge | Solution |
|-----------|----------|
| Irrelevant retrieval | Improve chunking, add metadata filtering |
| Hallucinations | Enforce context grounding in prompts |
| Latency | Use smaller embedding models, caching |
| Cost | Batch API calls, selective document processing |

### 5. **Next Steps**
- **Multi-query**: Expand retrieval with query reformulation
- **Filtering**: Add metadata filters for precision
- **Ranking**: Score retrieved documents by relevance
- **Routing**: Direct queries to appropriate document sets

This foundation prepares you for advanced retrieval strategies in Segment 2!